In [ ]:
using Pkg
using Random
using Statistics
using Printf
using LinearAlgebra
using Logging

function find_project_root(start::AbstractString=pwd())
    active_project = Base.active_project()
    if !isnothing(active_project)
        active_root = dirname(active_project)
        if isfile(joinpath(active_root, "Project.toml")) && isfile(joinpath(active_root, "src", "System1D.jl"))
            return active_root
        end
    end

    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "System1D.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate project root from $start")
        dir = parent
    end
end

PROJECT_ROOT = find_project_root()
Pkg.activate(PROJECT_ROOT; io=devnull)

using Revise
using Plots

includet(joinpath(PROJECT_ROOT, "Experiments", "common", "notebook_helpers.jl"))

NOTEBOOK_REL_DIR = joinpath("Experiments", "systems", "periodic_ion_ring_1d", "gfmc", "notebooks")
PATHS = nb_paths(PROJECT_ROOT, NOTEBOOK_REL_DIR)
nb_include_formatting(PATHS.notebook_dir)

includet(joinpath(PROJECT_ROOT, "src", "System1D.jl"))
using .System1D
includet(joinpath(PATHS.notebook_dir, "periodic_ion_ring_helpers.jl"))

default(; dpi=170)
nothing


## Bosons on a Periodic Ion Ring: Tonks-Girardeau `|det|` Scaffold

This notebook builds a guided bosonic GFMC run using a positive Tonks-Girardeau scaffold

`Ψ_T(R; X) = |det[ϕ_α(x_i; X)]| * exp(sum_{i<j} w_pair(x_i, x_j))`

where:
- `ϕ_α(x; X)` are the lowest one-body orbitals of the ionic external potential for the current ion positions `X`
- `|det|` supplies the fermionized hard-core/TG structure while staying bosonic and nonnegative
- `w_pair` is an optional smooth positive pair correction on top of the TG scaffold
- GFMC runs with `ImportanceGuiding(trial, H)`, `NoNode()`, and `run_gfmc_with_vmc_init(...)`

Edit `N`, `M`, `L`, and `ion_positions` in the next code cell to change the geometry.
Set `ion_positions = nothing` to use equally spaced ions, or provide a length-`M` vector of ring positions.
If you increase `N`, keep `kmax` large enough that the one-body basis resolves at least the lowest `N` orbitals.


In [ ]:
N = 3
M = 3
a = 1.0
L = M * a

# Use `nothing` for equally spaced ions, or provide a length-M vector.
ion_positions = nothing
# ion_positions = [0.0, 0.92, 2.05]

D = 0.5
ion_strength = 1.0
ion_softening = 0.35 * a

bb_strength = 0.45
bb_softening = 0.25 * a
hard_core_radius = 0.0
hard_core_barrier = 0.0
pair_metric = :chord

kmax = max(6, N + 2)
quad_points = 1536

problem = build_periodic_ion_ring_boson_problem(
    N,
    M,
    a;
    L=L,
    ion_positions=ion_positions,
    D=D,
    ion_strength=ion_strength,
    ion_softening=ion_softening,
    bb_strength=bb_strength,
    bb_softening=bb_softening,
    hard_core_radius=hard_core_radius,
    hard_core_barrier=hard_core_barrier,
    pair_metric=pair_metric,
    kmax=kmax,
    quad_points=quad_points,
)

H = bosonic_hamiltonian(problem)

TRIAL_NODE_TOL = 1.0e-10
TRIAL_SMOOTH_PAIR_STRENGTH = 0.04
TRIAL_SMOOTH_PAIR_SOFTENING = max(bb_softening, 0.25 * a)

trial = bosonic_tg_scaffold_trial_wavefunction(
    problem;
    node_tol=TRIAL_NODE_TOL,
    smooth_pair_strength=TRIAL_SMOOTH_PAIR_STRENGTH,
    smooth_pair_softening=TRIAL_SMOOTH_PAIR_SOFTENING,
)
guiding = ImportanceGuiding(trial, H)

HAS_PAIR_OBSERVABLES = N >= 2
occupied_orbitals = collect(1:N)
occupied_energies = problem.ring.energies[occupied_orbitals]
noninteracting_boson_ref = noninteracting_boson_energy(problem.ring, N)
tg_scaffold_ref = tg_scaffold_energy(problem.ring, N)

targetN = 256

vmc_dt = 2.0e-3
vmc_nsteps = 80
vmc_ET0 = tg_scaffold_ref
vmc_params = VMCParams(; dt=vmc_dt, nsteps=vmc_nsteps, targetN=targetN, ET0=vmc_ET0)

gfmc_dt = 1.0e-3
gfmc_nsteps = 300
gfmc_nequil = 60
gfmc_ET0 = tg_scaffold_ref
feedback = 0.05
reconfiguration_interval = 2
branch_cap = 5.0
energy_window = 20
gfmc_params = GFMCParams(gfmc_dt, gfmc_nsteps, gfmc_nequil, targetN, gfmc_ET0, feedback, reconfiguration_interval, branch_cap, energy_window)

rng_init = MersenneTwister(1234)
warm_start_min_separation = max(hard_core_radius, 0.04 * a)
initial_positions = sample_boson_ring_configurations(problem, targetN, rng_init; min_separation=warm_start_min_separation)

MODEL_GRID_POINTS = 600
DENSITY_GRID_POINTS = 400
DENSITY_BANDWIDTH = 0.08 * a
PAIR_SEP_BINS = 80

xgrid_model = Float64[i * (problem.ring.L / MODEL_GRID_POINTS) for i in 0:(MODEL_GRID_POINTS - 1)]
rgrid_pair = collect(range(0.0, 0.5 * problem.ring.L; length=MODEL_GRID_POINTS))

onebody_potential_curve = Float64[onebody_potential(problem.ring, x) for x in xgrid_model]
dx_model = problem.ring.L / MODEL_GRID_POINTS
tg_density_reference = occupied_pooled_density(problem.ring, occupied_orbitals, xgrid_model; per_particle=true)
tg_density_reference ./= (sum(tg_density_reference) * dx_model)
pair_trial_factor_curve_vals = HAS_PAIR_OBSERVABLES ?
    smooth_pair_factor_curve(
        problem.ring.L,
        rgrid_pair;
        smooth_pair_strength=TRIAL_SMOOTH_PAIR_STRENGTH,
        smooth_pair_softening=TRIAL_SMOOTH_PAIR_SOFTENING,
    ) : ones(length(rgrid_pair))

SNAPSHOT_STEPS = nb_default_snapshot_steps(gfmc_nsteps)
ION_MARKERS = sort(problem.ring.ion_positions)
NSHOW_ORBITALS = min(N, 6)

RUN_LABEL = "bosonic TG scaffold"
RUN_COLOR = :navy
PLOT_TITLE = "Periodic ion ring GFMC (bosons, TG scaffold)"
MODEL_TRIAL_TITLE = "Bosonic |det|-scaffold diagnostics"
DENSITY_TITLE = "Periodic ion ring GFMC: pooled boson density evolution (TG scaffold)"
UNPOOLED_DENSITY_TITLE = "Periodic ion ring GFMC: final per-particle densities (TG scaffold)"
PAIR_TITLE = HAS_PAIR_OBSERVABLES ?
    "Periodic ion ring GFMC: final pair-separation density" :
    "Periodic ion ring GFMC: pair-separation density (not defined for N = 1)"
PARTICLE_COLORS = [:navy, :darkorange, :teal, :crimson, :purple, :goldenrod, :deeppink, :forestgreen]

VMC_PROPOSAL = DriftGaussianProposal()
RECONFIGURATION = SystematicReconfiguration()

VMC_SHOW_PROGRESS = false
VMC_PROGRESS_EVERY = 0
VMC_DEBUG_MODE = false
VMC_DEBUG_EVERY = 10

SHOW_PROGRESS = false
PROGRESS_EVERY = 0
DEBUG_MODE = false
DEBUG_EVERY = 20

WRITE_RUN_CSV = false
CSV_FILENAME = "periodic_ion_ring_bosons_tg_scaffold_gfmc_vmc_init.csv"
SAVE_FIGURES = false
FIGURE_STEM = "periodic_ion_ring_bosons_tg_scaffold_gfmc_vmc_init"


In [ ]:
sim = run_gfmc_with_vmc_init(
    H,
    gfmc_params,
    initial_positions,
    trial,
    vmc_params;
    vmc_rng=MersenneTwister(41),
    gfmc_rng=MersenneTwister(52),
    proposal=VMC_PROPOSAL,
    guiding=guiding,
    nodepolicy=NoNode(),
    reconfiguration=RECONFIGURATION,
    vmc_show_progress=VMC_SHOW_PROGRESS,
    vmc_progress_every=VMC_PROGRESS_EVERY,
    vmc_progress_label="VMC warm start",
    vmc_debug=VMC_DEBUG_MODE,
    vmc_debug_every=VMC_DEBUG_EVERY,
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABEL,
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

start_idx = min(gfmc_params.nequil + 1, length(sim.energy_mean_history))
mean_energy, sem_energy = nb_mean_sem(sim.energy_mean_history[start_idx:end])

final_snapshot = nb_last_snapshot(sim)
final_pair_sep = Float64[]
for R in final_snapshot
    for i in 1:(length(R) - 1)
        for j in (i + 1):length(R)
            push!(final_pair_sep, pair_distance(problem, R[i], R[j]))
        end
    end
end

println("GFMC step 0 corresponds to the VMC warm-start ensemble.")
println("ion positions = ", problem.ring.ion_positions)
println("occupied orbital indices = ", occupied_orbitals)
println("occupied one-body energies = ", occupied_energies)
println(@sprintf("noninteracting boson reference N*ε0 = %.8f", noninteracting_boson_ref))
println(@sprintf("Tonks/TG scaffold reference sum εα = %.8f", tg_scaffold_ref))
println(@sprintf("%s mean energy after nequil=%d: %.8f +/- %.3e", RUN_LABEL, gfmc_params.nequil, mean_energy, sem_energy))
println("final fixed walker count = ", sim.population_history[end])
println(@sprintf("final mean weight = %.6f", sim.mean_weight_history[end]))
println(@sprintf("final effective population = %.2f", sim.effective_population_history[end]))
if HAS_PAIR_OBSERVABLES
    println(@sprintf("minimum final pair separation = %.6f", minimum(final_pair_sep)))
else
    println("single-particle run: no pair separations to report.")
end

if WRITE_RUN_CSV
    csv_path = joinpath(PATHS.tables_dir, CSV_FILENAME)
    nb_write_csv(csv_path, nb_gfmc_rows(RUN_LABEL, sim))
    println("Wrote run CSV to: ", abspath(csv_path))
end


In [ ]:
function pooled_ring_coordinates(snapshot)
    xs = Float64[]
    for R in snapshot
        append!(xs, Float64.(R))
    end
    return xs
end

function ring_particle_coordinates(snapshot, particle_idx::Integer)
    idx = Int(particle_idx)
    return Float64[R[idx] for R in snapshot]
end

function pair_separations(snapshot)
    rs = Float64[]
    for R in snapshot
        for i in 1:(length(R) - 1)
            for j in (i + 1):length(R)
                push!(rs, pair_distance(problem, R[i], R[j]))
            end
        end
    end
    return rs
end

step0_snapshot = sim.walker_positions_history[1]
step0_xs = pooled_ring_coordinates(step0_snapshot)
step0_centers, step0_density = nb_periodic_kde_curve(
    step0_xs;
    xmin=0.0,
    xmax=problem.ring.L,
    grid_points=DENSITY_GRID_POINTS,
    bandwidth=DENSITY_BANDWIDTH,
)

final_snapshot = nb_last_snapshot(sim)
final_xs = pooled_ring_coordinates(final_snapshot)
final_centers, final_density = nb_periodic_kde_curve(
    final_xs;
    xmin=0.0,
    xmax=problem.ring.L,
    grid_points=DENSITY_GRID_POINTS,
    bandwidth=DENSITY_BANDWIDTH,
)

p_potential = plot(
    xgrid_model,
    onebody_potential_curve;
    xlabel="x",
    ylabel="V_IB(x)",
    title="Ionic external potential",
    color=:black,
    linewidth=2.4,
    label="V_IB(x)",
    xlims=(0.0, problem.ring.L),
)
for (k, xmark) in enumerate(ION_MARKERS)
    vline!(p_potential, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion positions" : ""))
end

p_orbitals = plot(
    xlabel="x",
    ylabel="density",
    title=(NSHOW_ORBITALS == N ? "Occupied one-body orbital densities" : "First $(NSHOW_ORBITALS) occupied orbital densities"),
    legend=:topright,
    xlims=(0.0, problem.ring.L),
)
for orb in 1:NSHOW_ORBITALS
    vals, _, _ = orbital_curve(problem.ring, orb, xgrid_model)
    density = vals .^ 2
    density ./= (sum(density) * dx_model)
    plot!(p_orbitals, xgrid_model, density; linewidth=2.2, label="|ϕ_$(orb)|²")
end
for (k, xmark) in enumerate(ION_MARKERS)
    vline!(p_orbitals, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion positions" : ""))
end

p_pair_trial = plot(
    rgrid_pair,
    pair_trial_factor_curve_vals;
    xlabel="r",
    ylabel="factor",
    title=(HAS_PAIR_OBSERVABLES ? "Additional smooth pair correction" : "Pair correction is trivial for N = 1"),
    color=:darkorange,
    linewidth=2.4,
    label=(HAS_PAIR_OBSERVABLES ? "exp(w_pair(r))" : "identity factor"),
    xlims=(0.0, 0.5 * problem.ring.L),
)

p_density_compare = plot(
    xlabel="x",
    ylabel="density",
    title="Warm-start and GFMC density vs TG scaffold reference",
    legend=:topright,
    xlims=(0.0, problem.ring.L),
)
plot!(p_density_compare, xgrid_model, tg_density_reference; color=:black, linewidth=2.2, linestyle=:dash, label="TG scaffold density")
plot!(p_density_compare, step0_centers, step0_density; color=RUN_COLOR, linewidth=2.4, label="step 0 density (after VMC)")
plot!(p_density_compare, final_centers, final_density; color=:crimson, linewidth=2.2, linestyle=:dot, label="final GFMC density")
for (k, xmark) in enumerate(ION_MARKERS)
    vline!(p_density_compare, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion positions" : ""))
end

model_fig = plot(p_potential, p_orbitals, p_pair_trial, p_density_compare; layout=(2, 2), size=(1400, 980), plot_title=MODEL_TRIAL_TITLE)
display(model_fig)
nb_save_figure(model_fig, PATHS.figures_dir, FIGURE_STEM, "model_trial"; enabled=SAVE_FIGURES)

history_fig = nb_plot_gfmc_history([sim]; labels=[RUN_LABEL], colors=[RUN_COLOR], title_prefix=PLOT_TITLE)
display(history_fig)
nb_save_figure(history_fig, PATHS.figures_dir, FIGURE_STEM, "history"; enabled=SAVE_FIGURES)

available_steps = SNAPSHOT_STEPS[1:min(length(SNAPSHOT_STEPS), length(sim.walker_positions_history))]
density_fig = plot(
    xlabel="x",
    ylabel="density",
    title=DENSITY_TITLE,
    legend=:topright,
    xlims=(0.0, problem.ring.L),
)
for (snapshot, step_idx) in zip(sim.walker_positions_history, available_steps)
    xs = pooled_ring_coordinates(snapshot)
    centers, density = nb_periodic_kde_curve(
        xs;
        xmin=0.0,
        xmax=problem.ring.L,
        grid_points=DENSITY_GRID_POINTS,
        bandwidth=DENSITY_BANDWIDTH,
    )
    step_label = step_idx == 0 ? "step 0 (after VMC warm start)" : "step $(step_idx)"
    plot!(density_fig, centers, density; label=step_label, color=RUN_COLOR, linewidth=2.2, alpha=0.82)
end
plot!(density_fig, xgrid_model, tg_density_reference; color=:black, linewidth=2.0, linestyle=:dash, label="TG scaffold density")
for (k, xmark) in enumerate(ION_MARKERS)
    vline!(density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion positions" : ""))
end

display(density_fig)
nb_save_figure(density_fig, PATHS.figures_dir, FIGURE_STEM, "density"; enabled=SAVE_FIGURES)

particle_colors = [PARTICLE_COLORS[1 + mod(i - 1, length(PARTICLE_COLORS))] for i in 1:N]
unpooled_density_fig = plot(
    xlabel="x",
    ylabel="density",
    title=UNPOOLED_DENSITY_TITLE,
    legend=:topright,
    xlims=(0.0, problem.ring.L),
)
for particle_idx in 1:N
    xs = ring_particle_coordinates(final_snapshot, particle_idx)
    centers, density = nb_periodic_kde_curve(
        xs;
        xmin=0.0,
        xmax=problem.ring.L,
        grid_points=DENSITY_GRID_POINTS,
        bandwidth=DENSITY_BANDWIDTH,
    )
    plot!(unpooled_density_fig, centers, density; label="particle $(particle_idx)", color=particle_colors[particle_idx], linewidth=2.3)
end
for (k, xmark) in enumerate(ION_MARKERS)
    vline!(unpooled_density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion positions" : ""))
end

display(unpooled_density_fig)
nb_save_figure(unpooled_density_fig, PATHS.figures_dir, FIGURE_STEM, "density_unpooled"; enabled=SAVE_FIGURES)

if HAS_PAIR_OBSERVABLES
    final_pair_sep = pair_separations(final_snapshot)
    pair_centers, pair_density = nb_density_curve(
        final_pair_sep;
        nbins=PAIR_SEP_BINS,
        xmin=0.0,
        xmax=0.5 * problem.ring.L,
        smoothing_window=9,
    )
    pair_fig = plot(
        pair_centers,
        pair_density;
        xlabel="r",
        ylabel="density",
        title=PAIR_TITLE,
        color=:darkorange,
        linewidth=2.4,
        label=false,
    )
else
    pair_fig = plot(
        xlabel="r",
        ylabel="density",
        title=PAIR_TITLE,
        legend=false,
        xlims=(0.0, 0.5 * problem.ring.L),
        ylims=(0.0, 1.0),
    )
    annotate!(pair_fig, 0.25 * problem.ring.L, 0.5, Plots.text("No pair-separation density for N = 1", 11, :darkorange))
end

display(pair_fig)
nb_save_figure(pair_fig, PATHS.figures_dir, FIGURE_STEM, "pair_density"; enabled=SAVE_FIGURES)
